# Inference and Serving with alignrl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sacredvoid/alignrl/blob/main/notebooks/05_inference_serving.ipynb)

This notebook covers **inference optimization and serving** for your post-trained models. After training (SFT, GRPO, DPO), you need to actually use the model. alignrl provides a unified `ModelServer` that supports multiple backends: Unsloth (default, good for development), vLLM (production-grade throughput), and MLX (Apple Silicon). We'll also launch an interactive Gradio demo that lets you compare outputs across training stages.

## Setup

Install alignrl with inference and Gradio dependencies.

In [ ]:
!pip install alignrl[train,unsloth,gradio] -q

## Inference Backends

alignrl supports three inference backends, each with different trade-offs:

| Backend | Best For | Speed | Memory | Platforms |
|---------|---------|-------|--------|-----------|
| **Unsloth** | Development, Colab | Good | Low (4-bit) | CUDA GPUs |
| **vLLM** | Production serving | Excellent | Medium | CUDA GPUs |
| **MLX** | Local development | Good | Low | Apple Silicon |

### How the Backends Work

- **Unsloth**: Loads the base model in 4-bit quantization, merges LoRA adapters, and runs standard HuggingFace `generate()`. Optimized kernels make this faster than vanilla transformers.
- **vLLM**: Uses PagedAttention and continuous batching for high-throughput serving. Ideal when you need to serve many requests concurrently.
- **MLX**: Apple's ML framework for Apple Silicon Macs. Runs on the unified memory architecture (CPU+GPU share memory), making it memory-efficient for local use.

## Loading a Model

The `ModelServer` class provides a unified interface. You specify the base model, an optional adapter path, and the backend.

In [ ]:
from alignrl.inference import InferenceConfig, ModelServer, build_prompt

# Configuration for the Unsloth backend (works on Colab T4)
config = InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path=None,  # Set to "./outputs/grpo/final" if you have a trained adapter
    backend="unsloth",
    temperature=0.7,
    max_tokens=512,
    top_p=0.9,
)

print(f"Backend: {config.backend}")
print(f"Model: {config.model_name}")
print(f"Adapter: {config.adapter_path or 'None (base model)'}")
print(f"Temperature: {config.temperature}")
print(f"Max tokens: {config.max_tokens}")

In [ ]:
import time

server = ModelServer(config)

start = time.time()
server.load()
load_time = time.time() - start

print(f"Model loaded in {load_time:.1f}s")

## Generating Text

The `build_prompt` helper creates a properly formatted chat message list. The `generate` method handles tokenization, generation, and decoding.

In [ ]:
# Simple generation
messages = build_prompt(
    "What are the three laws of robotics?",
    system="You are a helpful, concise assistant."
)

print("Prompt messages:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:100]}")

print("\nGenerating...")
start = time.time()
response = server.generate(messages)
gen_time = time.time() - start

print(f"\nResponse ({gen_time:.2f}s):")
print(response)

## Throughput Benchmarking

Let's measure how fast the model generates tokens. This is important for understanding serving costs and latency.

In [ ]:
import time

benchmark_prompts = [
    "Explain how a neural network learns in 3 sentences.",
    "Write a Python function to check if a number is prime.",
    "What is the capital of France and why is it significant?",
    "Summarize the theory of relativity in simple terms.",
    "List 5 tips for writing clean code.",
]

total_tokens = 0
total_time = 0
results = []

for prompt_text in benchmark_prompts:
    msgs = build_prompt(prompt_text, system="You are a helpful assistant.")

    start = time.time()
    response = server.generate(msgs)
    elapsed = time.time() - start

    # Rough token count (words * 1.3 as approximation)
    approx_tokens = int(len(response.split()) * 1.3)
    tokens_per_sec = approx_tokens / elapsed if elapsed > 0 else 0

    total_tokens += approx_tokens
    total_time += elapsed

    results.append({
        "prompt": prompt_text[:50],
        "tokens": approx_tokens,
        "time": elapsed,
        "tok_per_sec": tokens_per_sec,
    })

print(f"{'Prompt':<52} {'Tokens':>7} {'Time':>7} {'Tok/s':>7}")
print("-" * 75)
for r in results:
    print(f"{r['prompt']:<52} {r['tokens']:>7} {r['time']:>6.2f}s {r['tok_per_sec']:>6.1f}")

print("-" * 75)
avg_tps = total_tokens / total_time if total_time > 0 else 0
print(f"{'TOTAL':<52} {total_tokens:>7} {total_time:>6.2f}s {avg_tps:>6.1f}")
print(f"\nAverage throughput: {avg_tps:.1f} tokens/sec")

## Loading Trained Adapters

If you've run the previous training notebooks, you can load those adapters for comparison. Let's check what's available and load any that exist.

In [ ]:
from pathlib import Path

available_stages = {"base": None}
stage_paths = {
    "sft": "./outputs/sft/final",
    "grpo": "./outputs/grpo/final",
    "dpo": "./outputs/dpo/final",
}

for stage, path in stage_paths.items():
    if Path(path).exists():
        available_stages[stage] = path
        print(f"[found] {stage}: {path}")
    else:
        print(f"[missing] {stage}: {path}")

print(f"\nAvailable stages for comparison: {list(available_stages.keys())}")

## Interactive Gradio Demo

The alignrl `create_demo` function builds a Gradio interface that loads all available training stages and lets you compare their outputs side by side. This runs inline in Colab.

In [ ]:
from alignrl.demo import create_demo

# Create the demo with all available stages
demo = create_demo(
    stages=available_stages,
    model_name="Qwen/Qwen2.5-3B",
)

# Launch inline in Colab
demo.launch(share=True)

## Manual Stage Comparison

If you prefer non-interactive comparison, here's how to load multiple stages and compare their outputs programmatically:

In [ ]:
# Load servers for each available stage
servers = {}
for stage, adapter_path in available_stages.items():
    cfg = InferenceConfig(
        model_name="Qwen/Qwen2.5-3B",
        adapter_path=adapter_path,
        backend="unsloth",
        max_tokens=256,
        temperature=0.7,
    )
    srv = ModelServer(cfg)
    srv.load()
    servers[stage] = srv
    print(f"Loaded {stage} model")

# Compare on a math problem
test_questions = [
    "What is 25% of 360?",
    "A book costs $15. If you buy 3 books and pay with a $50 bill, how much change do you get?",
]

MATH_SYSTEM = (
    "You are a helpful math assistant. Solve problems step by step, "
    "showing your reasoning clearly. Put your final answer in \\boxed{}."
)

for question in test_questions:
    print(f"\nQ: {question}")
    print("=" * 60)
    messages = build_prompt(question, system=MATH_SYSTEM)
    for stage, srv in servers.items():
        response = srv.generate(messages)
        print(f"\n  [{stage}]:")
        print(f"  {response[:300]}")
    print()

## Using Other Backends

Here's how you would configure vLLM or MLX backends (these won't run on Colab T4 but are useful for reference):

### vLLM (Production Serving)
```python
# Requires: pip install alignrl[serve]
config = InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path="./outputs/grpo/final",
    backend="vllm",
    temperature=0.7,
    max_tokens=512,
)
server = ModelServer(config)
server.load()  # Starts the vLLM engine
```

### MLX (Apple Silicon)
```python
# Requires: pip install alignrl[mlx]
config = InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path="./outputs/grpo/final",
    backend="mlx",
    temperature=0.7,
    max_tokens=512,
)
server = ModelServer(config)
server.load()
```

## Save Results Summary

In [ ]:
import json

# Save throughput benchmark results
benchmark_summary = {
    "backend": config.backend,
    "model": config.model_name,
    "adapter": config.adapter_path,
    "total_tokens": total_tokens,
    "total_time_seconds": round(total_time, 2),
    "avg_tokens_per_sec": round(avg_tps, 1),
    "num_prompts": len(benchmark_prompts),
}

output_path = Path("./results/inference_benchmark.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(benchmark_summary, f, indent=2)

print(f"Benchmark results saved to {output_path}")
print(json.dumps(benchmark_summary, indent=2))

## Summary

This completes the alignrl notebook series. Here's what we covered:

| Notebook | Technique | What It Does |
|----------|-----------|-------------|
| 01 | SFT | Teaches the model to follow instructions |
| 02 | GRPO | Reinforces math reasoning with verifiable rewards |
| 03 | DPO | Aligns outputs with human preferences |
| 04 | Evaluation | Benchmarks all stages on standard tasks |
| 05 | Inference | Serves and demos the trained model |

All adapters are saved as lightweight LoRA weights (~50MB each) that can be loaded on top of the base Qwen2.5-3B model. For production use, consider merging the adapters into the base model and serving with vLLM.